In [11]:
import os
import pandas as pd
from pathlib import Path
from datetime import datetime

def get_file_info(base_path, version_suffix):
    base_path = Path(base_path)
    data = []
    
    # 해당 폴더 내 모든 .md 파일 검색
    for file_path in base_path.rglob("*.md"):
        # 파일명에서 버전 접미사 제거하여 비교용 키 생성 (예: '1. 공리_v92' -> '1. 공리')
        clean_name = file_path.stem.replace(f'_{version_suffix}', '')
        
        # 정보 수집
        stats = file_path.stat()
        mtime = datetime.fromtimestamp(stats.st_mtime).strftime('%Y-%m-%d %H:%M:%S')
        
        # 라인 수 계산
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                line_count = sum(1 for _ in f)
        except:
            line_count = 0

        data.append({
            '파일명': clean_name,
            f'크기_{version_suffix}': round(stats.st_size / 1024, 2), # KB
            f'길이_{version_suffix}': line_count,
            f'수정일시_{version_suffix}': mtime
        })
    
    return pd.DataFrame(data)

In [14]:
v1='v91'
v2='v92'

# 1. 경로 설정
v91_path = "./essay_"+v1
v92_path = "./essay_"+v2

# 2. 각 버전 데이터 수집
df_91 = get_file_info(v91_path, v1)
df_92 = get_file_info(v92_path, v2)

# 3. '파일명'을 기준으로 병합 (Outer Join을 통해 한쪽에만 있는 파일도 표시)
df_compare = pd.merge(df_91, df_92, on='파일명', how='outer')

# 4. 보기 좋게 정렬 (파일명 기준)
df_compare = df_compare.sort_values('파일명').reset_index(drop=True)

# 결과 확인
df_compare


In [16]:
from openpyxl.styles import PatternFill, Font, Alignment
from datetime import datetime

excel_file = "essay_comparison.xlsx"

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    df_compare.to_excel(writer, index=False, sheet_name='비교결과')
    
    workbook = writer.book
    worksheet = writer.sheets['비교결과']
    
    # 스타일 정의
    latest_fill = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid') # 연한 녹색 (최신)
    header_fill = PatternFill(start_color='DDEBF7', end_color='DDEBF7', fill_type='solid')
    
    # 헤더 스타일
    for cell in worksheet[1]:
        cell.fill = header_fill
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center')
    
    # 데이터 행 반복
    for row_num in range(2, len(df_compare) + 2):
        val_91 = worksheet.cell(row=row_num, column=4).value
        val_92 = worksheet.cell(row=row_num, column=7).value
        
        try:
            # 문자열을 datetime 객체로 변환하여 비교
            dt_91 = datetime.strptime(val_91, '%Y-%m-%d %H:%M:%S') if val_91 else None
            dt_92 = datetime.strptime(val_92, '%Y-%m-%d %H:%M:%S') if val_92 else None
            
            if dt_91 and dt_92:
                if dt_91 > dt_92:
                    # v91이 더 최신인 경우
                    worksheet.cell(row=row_num, column=4).fill = latest_fill
                elif dt_92 > dt_91:
                    # v92가 더 최신인 경우
                    worksheet.cell(row=row_num, column=7).fill = latest_fill
            elif dt_91: # 한쪽에만 데이터가 있는 경우 (그 자체가 최신)
                worksheet.cell(row=row_num, column=4).fill = latest_fill
            elif dt_92:
                worksheet.cell(row=row_num, column=7).fill = latest_fill
                
        except (ValueError, TypeError):
            # 날짜 형식이 아니거나 데이터가 없는 경우 스킵
            pass

    # 열 너비 자동 조절
    for col in worksheet.columns:
        max_length = 0
        column_letter = col[0].column_letter
        for cell in col:
            if cell.value:
                max_length = max(max_length, len(str(cell.value)))
        worksheet.column_dimensions[column_letter].width = max_length + 2

print(f"✅ 최신 날짜 강조 엑셀 저장 완료: {excel_file}")

✅ 최신 날짜 강조 엑셀 저장 완료: essay_comparison_latest_highlight.xlsx
